**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**01 | Original Dataset, Source Integrity and Trial Provenance**

![Original Essex source integrity and trial provenance](../../../images/01_consolidated_workflow.svg)

**Research question:** Are the original six-activity Essex source files authentic, participant-linked and traceable to unique movement records and original trial identifiers?

**Published source:** Hollinger et al. (2026), *Scientific Data*, [DOI 10.1038/s41597-026-07429-3](https://doi.org/10.1038/s41597-026-07429-3); [Figshare 30225739, version 13](https://doi.org/10.6084/m9.figshare.30225739.v13). This is the original full MATLAB dataset, not the separate 444-record BoneNet derivative.

**Research sequence:** Source identity and checksum → safe ZIP extraction → 40 original participant files → MATLAB Data fields and FE representations → workbook subject reconciliation → 1,897 original movement trials → activity/source coverage → integrated scientific Quality Gate.

**Previously recorded results:** 40 MAT files, 1,897 source Data elements, 1,897 readable UpdatedTrialID values, 168 source-empty FE fields and 1,729 opaque/structured FE fields. These are *historical* findings; this revised notebook must be executed again before claiming newly verified results.

**Scope and reproducibility:** Original record identifiers are preserved. Physical IMU/FE units, decoded strain waveforms, temporal alignment, clinical use and subject-independent regression are separate research questions. Historical 01/02 notebooks and all their outputs remain unchanged; new evidence is isolated in `output/_consolidation_drafts/01_source_and_trial/`.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.split("/")[-1].removesuffix(".git")

%cd /content

if not repo_path.exists():
    !git clone -q {repo_url}

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Environment**


In [ ]:
# ======================== 01. Imports and Environment ========================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

# ---------------------------- 2. Standard Library ----------------------------
import json
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4))

# ------------------ 5. Shared and Study Research Utilities -------------------
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ARTICLE_ID, ARTICLE_VERSION, fetch_version_manifest,
    find_published_file, ensure_source_file, inspect_matlab_archive,
    extract_matlab_files, inspect_subject_mat,
)
from essex_record_audit import (
    audit_workbook_ids, audit_mat_records, summarise_trial_evidence,
)

# ---------------------- 6. Source and Output Locations -----------------------
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
historical_output_dir = root / "biomechanical_load_estimation" / "output"
output_dir = historical_output_dir / "_consolidation_drafts" / "01_source_and_trial"
raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
assert output_dir != historical_output_dir
print("Figshare article:", ARTICLE_ID, "| version:", ARTICLE_VERSION)
print("Isolated new evidence:", output_dir)


**02. Official Figshare Source Manifest**

Read the version-pinned published article, file IDs, original download names, sizes, licence and published hashes. The actual archive integrity is checked independently in the next section.


In [ ]:
# =================== 02. Official Figshare Source Manifest ===================

# ------------------- 1. Retrieve Official Version Metadata -------------------
article, source_files = fetch_version_manifest()
archive_file = find_published_file(source_files, "MATLAB.zip")
source_table = pd.DataFrame([
    {
        "file_id": item.get("id"),
        "filename": item.get("name"),
        "size_bytes": item.get("size"),
        "download_url": item.get("download_url"),
        "published_md5": item.get("computed_md5") or item.get("supplied_md5"),
    }
    for item in source_files
])
display(source_table)

# -------------------- 2. Record Original Dataset Identity --------------------
metadata = {
    "article_id": ARTICLE_ID,
    "version": ARTICLE_VERSION,
    "title": article.get("title"),
    "doi": article.get("doi"),
    "published_date": article.get("published_date"),
    "license": article.get("license"),
    "archive_file_id": archive_file["id"],
    "archive_filename": archive_file["name"],
    "archive_size_bytes": archive_file["size"],
    "archive_published_md5": archive_file.get("computed_md5") or archive_file.get("supplied_md5"),
}
source_table.to_csv(output_dir / "01_figshare_file_manifest.csv", index=False)
print("Article:", metadata["title"])
print("Pinned archive:", metadata["archive_filename"])
print("Archive size (MiB):", round(metadata["archive_size_bytes"] / 1024**2, 2))


**03. Source Archive and ZIP Integrity**

Verify the official source archive using published file metadata and local SHA256, inspect ZIP structure and CRC, and prevent unsafe extraction.


In [ ]:
# =================== 03. Source Archive and ZIP Integrity ====================

# --------------- 1. Download and Verify Official Source Bytes ----------------
archive_path, archive_hashes = ensure_source_file(archive_file, raw_dir)
print("Archive:", archive_path)
print("Verified local SHA256:", archive_hashes["sha256"])

# --------------------- 2. Validate the Complete Archive ----------------------
zip_records = inspect_matlab_archive(archive_path, expected_mat_files=40)
archive_table = pd.DataFrame(zip_records)
display(archive_table.head(10))
archive_table.to_csv(output_dir / "01_matlab_archive_inventory.csv", index=False)
print("Verified MATLAB files:", int(archive_table["is_matlab_file"].sum()))
print("Total uncompressed MiB:", round(archive_table["size_bytes"].sum() / 1024**2, 2))


**04. Original Participant File Inventory**

Extract the 40 published participant MAT files and preserve their authentic, nonconsecutive identifiers. Check source file integrity and inventory.


In [ ]:
# ================== 04. Original Participant File Inventory ==================

# -------------------- 1. Extract Validated MATLAB Members --------------------
mat_paths = extract_matlab_files(archive_path, mat_dir, zip_records)
participant_ids = [path.stem.lower() for path in mat_paths]
assert len(mat_paths) == len(set(participant_ids)) == 40
assert all(path.stat().st_size > 0 for path in mat_paths)

# ------------------- 2. Inspect Explicit Source Filenames --------------------
participant_table = pd.DataFrame({
    "source_file": [path.name for path in mat_paths],
    "subject_id": participant_ids,
    "file_size_bytes": [path.stat().st_size for path in mat_paths],
})
display(participant_table)
participant_table.to_csv(output_dir / "01_matlab_participant_inventory.csv", index=False)
print("Independent source files:", len(participant_table))
print("IDs are observed from filenames, not generated by range.")


**05. Original MATLAB Source Schema**

Record original Data element counts, source-file SHA256 and MATLAB field representation without inventing decoded numeric strain waveforms.


In [ ]:
# ===================== 05. Original MATLAB Source Schema =====================

# --------------------- 1. Inspect Every Source MAT File ----------------------
audit_rows = []
for source_path in mat_paths:
    audit_rows.append(inspect_subject_mat(source_path))

audit_table = pd.DataFrame(audit_rows).sort_values("subject_id").reset_index(drop=True)
display(audit_table[[
    "subject_id", "record_count", "source_data_shape",
    "first_record_type", "field_names_first_record"
]].head(12))

# ---------------- 2. Save Source Representation and Coverage -----------------
audit_table.to_csv(output_dir / "01_matlab_record_schema_audit.csv", index=False)
print("Participants audited:", len(audit_table))
print("Total MATLAB Data records:", int(audit_table["record_count"].sum()))
print("Record counts by participant:")
display(audit_table[["subject_id", "record_count"]])


**06. Original FE Field Representation Census**

Inspect e1e3 source fields per participant and classify source-empty, opaque and numeric candidate representations. An opaque MATLAB object is not equivalent to a validated FE waveform.


In [ ]:
# ================ 06. Original FE Field Representation Census ================

# ------------------ 1. Summarise FE Target Representations -------------------
status_cols = [
    "numeric_candidate", "empty", "missing_field",
    "nonfinite_numeric", "matlab_object_unresolved",
    "non_numeric_unresolved",
]
target_table = audit_table[["subject_id", "record_count", *status_cols]].copy()
target_table.to_csv(output_dir / "01_femoral_strain_representation.csv", index=False)
display(target_table)
print("FE representation totals:")
display(target_table[status_cols].sum().rename("records").to_frame())


**07. Participant Workbook and Identity Reconciliation**

Read only original participant IDs from SubCharacteristics.xlsx (the source header is Participent ID), excluding workbook summary rows; verify one-to-one agreement with authentic MAT filenames. Do not expose the workbook's personal health notes.


In [ ]:
# =========== 07. Participant Workbook and Identity Reconciliation ============

# --------------- 1. Retrieve the Original Participant Workbook ---------------
workbook_item = find_published_file(source_files, "SubCharacteristics.xlsx")
workbook_path, workbook_hashes = ensure_source_file(workbook_item, raw_dir)

# ----------- 2. Reconcile the 40 Original Participant Identifiers ------------
mat_ids = sorted(participant_ids)
workbook_audit, verified_subjects = audit_workbook_ids(workbook_path, mat_ids)
display(pd.DataFrame([{
    "recognised_rows": workbook_audit["recognised_participant_rows"],
    "unique_ids": workbook_audit["unique_participant_ids"],
    "duplicate_ids": workbook_audit["duplicate_ids"],
    "source_id_match": workbook_audit["id_alignment_passed"],
}]))
print("Workbook IDs missing:", workbook_audit["mat_ids_not_in_workbook"])
print("Unexpected workbook IDs:", workbook_audit["workbook_ids_not_in_mat"])

# ------------- 3. Enforce and Export a Minimal Subject Registry --------------
assert workbook_audit["id_alignment_passed"]
assert workbook_audit["unique_participant_ids"] == 40
assert workbook_audit["recognised_participant_rows"] == 40
assert workbook_audit["duplicate_ids"] == 0
verified_subjects.to_csv(output_dir / "01_workbook_subject_ids.csv", index=False)
print("Source participant identity gate: PASSED")


**08. Original Trial Identity and Record Registry**

Audit all 1,897 Data elements with original UpdatedTrialID, task, subtask, participant and file-local index; preserve unique source keys and check participant/trial duplication.


In [ ]:
# ============== 08. Original Trial Identity and Record Registry ==============

# --------------------- 1. Inspect All Source MAT Records ---------------------
records = audit_mat_records(mat_paths)
trial_summary = summarise_trial_evidence(records)
display(records[[
    "subject_id", "local_record_index", "task", "subtask",
    "updated_trial_id", "imu_representation", "fe_representation"
]].head(18))

print("Audited source records:", trial_summary["source_records"])
print("Participants:", trial_summary["source_participants"])
print("Readable UpdatedTrialID fields:", trial_summary["readable_updated_trial_ids"])

# --------------------- 2. Save Original Record Registry ----------------------
records.to_csv(
    output_dir / "01_original_record_registry.csv", index=False
)
print("Duplicate participant/trial IDs:", trial_summary["duplicate_subject_trial_ids"])
print("No inferred source-trial identifiers were created.")


**09. Original IMU and FE Field Representation**

Describe the original IMU and FE representation at the record level; distinguish empty FE source fields from opaque objects. Do not equate structural representations with numeric signal validation.


In [ ]:
# =============== 09. Original IMU and FE Field Representation ================

# -------------- 1. Group Source Representations by Participant ---------------
coverage = (
    records.groupby(
        ["subject_id", "imu_representation", "fe_representation"]
    ).size().rename("records").reset_index()
)
coverage.to_csv(
    output_dir / "01_modality_representation.csv", index=False
)
display(coverage.head(18))

# ------------------- 2. Report Source Encoding Categories --------------------
encoding = pd.DataFrame(
    [{"modality": "IMU", "representation": k, "records": int(v)}
     for k, v in trial_summary["imu_encoding_counts"].items()]
    + [{"modality": "FE", "representation": k, "records": int(v)}
       for k, v in trial_summary["fe_encoding_counts"].items()]
)
display(encoding)
encoding.to_csv(
    output_dir / "01_modality_encoding_summary.csv", index=False
)
print("FE waveform units and temporal grid: NOT VERIFIED")


**10. Original Activity and FE Field Coverage**

Summarise all original activity categories and the distribution of FE source representations. This is descriptive coverage, not a model-performance assessment.


In [ ]:
# ================ 10. Original Activity and FE Field Coverage ================

# --------------- 1. Inspect Counts for Six Original Activities ---------------
activity_counts = records.groupby("task").size().sort_values(ascending=False)
display(activity_counts.rename("records").to_frame())

# ------------- 2. Describe Activity-Specific FE Source Coverage --------------
activity_coverage = (
    records.groupby(["task", "fe_representation"])
    .size().rename("records").reset_index()
)
activity_coverage.to_csv(output_dir / "01_activity_fe_coverage.csv", index=False)
display(activity_coverage)
assert len(activity_counts) == 6
assert activity_coverage["records"].sum() == len(records)


**11. Consolidated Research Coverage Figure**

Visualise participant, activity and FE source-field coverage in one figure, avoiding duplicate panels while preserving original figures as historical evidence.


In [ ]:
# ================= 11. Consolidated Research Coverage Figure =================

# ---------- 1. Prepare Participant and FE Representation Summaries -----------
subject_counts = records.groupby("subject_id").size().sort_index()
fe_counts = records["fe_representation"].value_counts()

# ---------- 2. Plot Three Complementary Source-Level Distributions -----------
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
positions = list(range(len(subject_counts)))
axes[0].bar(positions, subject_counts.values)
axes[0].set_xticks(positions[::4])
axes[0].set_xticklabels(subject_counts.index[::4], rotation=65)
axes[0].set_title("Original source records by participant")
axes[0].set_ylabel("Source records")
axes[1].barh(activity_counts.index, activity_counts.values)
axes[1].invert_yaxis()
axes[1].set_title("Original activity distribution")
axes[1].set_xlabel("Source records")
axes[2].barh(fe_counts.index, fe_counts.values)
axes[2].invert_yaxis()
axes[2].set_title("Source FE representations")
axes[2].set_xlabel("Source records")
save_figure(fig, output_dir / "01_combined_source_coverage.png")
plt.show()


**12. Integrated Source and Trial Quality Gate**

Combine the source-integrity checks of historical Notebook 01 and the record/participant checks of Notebook 02. Compare original JSON SHA256 evidence in read-only mode. The resulting report retains the published source metadata, per-file digests, subject agreement and trial registry, without claiming numeric FE modelling readiness.


In [ ]:
# =============== 12. Integrated Source and Trial Quality Gate ================

# ---------------- 1. Source, Subject and Trial Quality Checks ----------------
assert int(target_table["empty"].sum()) == 168
assert int(audit_table["record_count"].sum()) == 1897
assert len(records) == 1897 and records["local_record_key"].is_unique
assert records["task"].nunique() == 6
prior_01 = json.loads((historical_output_dir / "01_source_provenance_and_quality_gate.json").read_text())
prior_02 = json.loads((historical_output_dir / "02_record_provenance_and_quality_gate.json").read_text())
assert prior_01["archive_sha256"] == archive_hashes["sha256"] == prior_02["archive_sha256"]
assert prior_01["matlab_data_elements"] == prior_02["source_record_summary"]["source_records"] == len(records)

assert ARTICLE_ID == 30225739 and ARTICLE_VERSION == 13
assert workbook_audit["id_alignment_passed"]
assert trial_summary["source_participants"] == 40
assert trial_summary["source_records"] == 1897
assert records["local_record_key"].is_unique
assert set(records["subject_id"]) == set(mat_ids)
assert int(sum(trial_summary["fe_encoding_counts"].values())) == len(records)
assert int(sum(trial_summary["imu_encoding_counts"].values())) == len(records)

fe_numeric_count = trial_summary["fe_encoding_counts"].get(
    "numeric_candidate", 0
)

# ----------- 2. Verify Both Historical SHA256 and Record Manifests -----------
assert prior_01["participant_workbook_sha256"] == workbook_hashes["sha256"]
assert prior_02["participant_workbook_sha256"] == workbook_hashes["sha256"]
assert prior_01["per_file_sha256"] == dict(zip(
    audit_table["source_file"], audit_table["source_sha256"]
))
assert prior_01["fe_representation_counts"] == {
    name: int(target_table[name].sum()) for name in status_cols
}
assert prior_02["source_record_summary"] == trial_summary
assert prior_02["participant_workbook_audit"] == workbook_audit
assert trial_summary["readable_updated_trial_ids"] == 1897
assert trial_summary["duplicate_subject_trial_ids"] == 0
assert int(target_table["matlab_object_unresolved"].sum()) == 1729
assert sum(trial_summary["imu_encoding_counts"].values()) == 1897

# ------- 3. Preserve Original Source and Trial Provenance in One JSON --------
provenance = {
    "source_article_id": ARTICLE_ID,
    "version": ARTICLE_VERSION,
    "publication_title": metadata["title"],
    "publication_doi": metadata["doi"],
    "publication_license": metadata["license"],
    "published_archive_file_id": metadata["archive_file_id"],
    "published_archive_size_bytes": metadata["archive_size_bytes"],
    "published_archive_md5": metadata["archive_published_md5"],
    "archive_sha256": archive_hashes["sha256"],
    "archive_md5": archive_hashes["md5"],
    "participant_workbook_sha256": workbook_hashes["sha256"],
    "participant_workbook_audit": workbook_audit,
    "source_record_summary": trial_summary,
    "per_file_sha256": dict(zip(
        audit_table["source_file"], audit_table["source_sha256"]
    )),
    "fe_representation_counts": {
        name: int(target_table[name].sum()) for name in status_cols
    },
    "source_and_trial_quality_gate": "PASSED",
    "numeric_fe_waveforms_verified_here": False,
    "physical_units_verified_here": False,
    "imu_fe_timebase_verified_here": False,
    "model_ready_here": False,
    "model_training_performed": False,
}
(output_dir / "01_consolidated_source_and_trial_quality_gate.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8"
)
print("CONSOLIDATED 01 | SOURCE AND TRIAL QUALITY GATE: PASSED")
print("Historical research outputs overwritten: NO")
print("Regression model-readiness established: NO")


**13. Colab Output Validation and Download**

Check the 12 consolidated outputs before using the required standard downloader. That downloader recursively packages Stage 01 files, including historical 01_* evidence; the additional draft-only ZIP unambiguously contains only this notebook's 12 new results.


In [ ]:
# ================= 13. Colab Output Validation and Download ==================

# ------------- 1. Validate All 12 Consolidated Research Outputs --------------
expected = [
    "01_figshare_file_manifest.csv",
    "01_matlab_archive_inventory.csv",
    "01_matlab_participant_inventory.csv",
    "01_matlab_record_schema_audit.csv",
    "01_femoral_strain_representation.csv",
    "01_workbook_subject_ids.csv",
    "01_original_record_registry.csv",
    "01_modality_representation.csv",
    "01_modality_encoding_summary.csv",
    "01_activity_fe_coverage.csv",
    "01_combined_source_coverage.png",
    "01_consolidated_source_and_trial_quality_gate.json",
]
missing = [name for name in expected if not (output_dir / name).is_file()]
if missing:
    raise FileNotFoundError("Incomplete consolidated outputs: " + ", ".join(missing))
print("Validated consolidated evidence files:", len(expected))

# ------------- 2. Package Research Outputs for Manual Retrieval --------------
download_outputs_from_colab(folder="biomechanical_load_estimation/output", stage="01")

# --------------- 3. Package a Separate Draft-Only Research ZIP ---------------
archive = Path("/content") / "BoneNet_Consolidated_01_Draft_Outputs.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for name in expected:
        z.write(output_dir / name, arcname=name)
print("Draft-only Colab ZIP:", archive)
print("Notebook GitHub Commit/Push performed: NO")


**14. Research Interpretation and Next Stage Boundary**

**Research contribution:** A coherent, source-verified workflow for the official six-activity Essex dataset: archive and licence provenance, 40 authentic participant identifiers, original source MAT structures, 1,897 traceable movement records, original FE representation and missingness, source-specific coverage and two historically recorded Quality Gates.

**Historic findings to reconfirm after revision:** 1,897 readable original trial IDs with no duplicate participant–trial combinations; 1,897 IMU field representations, 1,729 opaque/structured FE representations and 168 source-empty FE fields. The revised code has no preserved execution outputs until it is run again.

**Scientific limitations:** Verified source integrity and original field representation are not the same as numerical FE waveform recovery, physical unit verification, time alignment, valid independent Test splitting or clinical generalisation.

**Evidence protection:** The original research files and executed notebooks remain unchanged. The 12 newly designed outputs are isolated in the consolidation directory and should be committed after inspection of an actual fresh Colab execution.

**Next independent topic:** MATLAB object decoding and numerical signal recovery, integrating historical Stages 03–05. No work on that notebook begins before this Notebook 01 revision is reviewed.
